In [1]:
# ===========================================================
# == This Jupyter notebook shows an example of using the   ==
# == covariate selection for linear models function in the ==
# == context of time-index dF/F data                       ==
# ==                                                       ==
# == Authors: Ethan Ancell (algorithm) & Madelyn Hjort     ==
# == See Hjort et al. 2026 for full description of methods ==
# ===========================================================

In [2]:
# Libraries
import numpy as np
import pandas as pd
from scipy.linalg import svd
import scipy.stats
from matplotlib import pyplot as plt
from sklearn import linear_model
import statsmodels.api as sm
import os
%matplotlib inline


# Load Files

Read in all predictor variables individually. This facilitates the leave-one-out method for determining significance.

The sample data includes:

<b>PREDICTORS</b>: Here I have them in csv format where row=time and col=kernel step
- 'cues' = cue delivery + trace interval
- 'lickRate' = licking shifted +/-300ms to account for preparatory and feedback activity
- 'R'= 1s starting at reward delivery (2.5-3.5s after cue)
- 'NR'=1s starting at no reward delivery (2.5-3.5s after cue)
- 'value' = the mRPE model value for the trial
- 'RPE' = the mRPE model rpe for the trial
- 'CD' = the mRPE model CD for the trial
- 'CE' = the mRPE model CE for the trial

<b>NEURAL ACTIVITY</b>: Here I have them in csv format where row=time and col=cell
- 'cells'= the neural activity for the session (7.5 Hz). This data is from the example animal in Fig. 2b, 2f&g

In [3]:
fdir=r'C:\Users\nsciebner\Desktop\PFC Revision\GLM sample data' # the file directory
os.chdir(fdir)

# Load predictors
X_cues = pd.read_csv('cues.csv').to_numpy()
X_lick = pd.read_csv('lickRate.csv').to_numpy()
X_R = pd.read_csv('R.csv').to_numpy()
X_NR = pd.read_csv('NR.csv').to_numpy()
X_value = pd.read_csv('value.csv').to_numpy()
X_RPE = pd.read_csv('rpe.csv').to_numpy()
X_CD = pd.read_csv('CD.csv').to_numpy()
X_CE = pd.read_csv('CE.csv').to_numpy()

# Load response vector (cell activity)
Y_all = pd.read_csv(r'Y_all.csv').to_numpy()

# Build predictor matrix in list style. For every chunk you want to test, include it
# as a component of the list.
X_full = [X_cues, X_lick, X_R, X_NR, X_value, X_RPE, X_CD, X_CE]

## Part 1: Function to create null Y


In [4]:
# Use  wrapping to get null Y
def get_ynull_wrapping(y_full, seed=1):
    
    T = y_full.shape[0]
    
    # Wrap function
    def wrap_y(y_array, q):
        if q != 0:
            return(np.hstack((y_array[q:y_array.shape[0]], 
                         y_array[0:q])))
        else:
            return(y_array)
        
    rng = np.random.default_rng(seed=seed)
    
    return_y = np.empty((y_full.shape[0], y_full.shape[1]))
    for i in range(y_full.shape[1]):
        q = rng.integers(0, T-1)
        return_y[:, i] = wrap_y(y_full[:, i], q)
    return return_y

In [5]:
# Create the null distribution
Y_null_wrap = get_ynull_wrapping(Y_all)

## Part 2: Function to get p-values / F-statistics using null Y data above

#### NOTE: This function is modified to also get $\Delta R^2$ and the wrapping $p$-values for comparison.

In [6]:
def compute_pvalues_null_data(X_list, Y_full, Y_null):
    
    # Error check input
    T = Y_null.shape[0]
    num_cells = np.shape(Y_null)[1]
    if Y_full.shape[0] != T or Y_full.shape[1] != Y_null.shape[1]:
        raise Exception('Y_full and Y_null should have same dimensions.')
    for xi, x_comp in enumerate(X_list):
        if x_comp.shape[0] != T:
            raise Exception('X_list at index ' + str(xi) + ' has ' + x_comp.shape[0] + ' time points but should have ' + str(T) + '.')
    
    # Compute null distribution of F-statistics
    num_cells_gen_ITI_F = Y_null.shape[1]
    f_stats_ITIs_allcells = np.empty((num_cells_gen_ITI_F, len(X_full))) # (cells in dataset, chunks)
    f_stats_observed = np.empty((num_cells, len(X_full)))
    
    # Build full matrix of X and get SVD
    K = len(X_list)
    X_full_func = np.ones(T).reshape(-1, 1)
    for X_c in X_list:
        if len(X_c.shape) == 1:
            X_full_func = np.hstack((X_full_func, X_c.reshape(-1, 1)))
        else:
            X_full_func = np.hstack((X_full_func, X_c))
    M = X_full_func.shape[1]
    U = svd(X_full_func, full_matrices=False)[0]
    
    # Optional: Also get Delta R2
    delta_r2 = np.empty((num_cells, K))
    
    # Loop through dropped chunks
    for k in range(K):
        # Build reduced matrix
        X_c = X_list[k]
        m_k = np.shape(X_c)[1]
        X_red = np.ones(T).reshape(-1, 1)
        for kt in range(K):
            if kt != k:
                X_ct = X_list[kt]
                if len(X_ct.shape) == 1:
                    X_red = np.hstack((X_red, X_ct.reshape(-1, 1)))
                else:
                    X_red = np.hstack((X_red, X_ct))
        U_red = svd(X_red, full_matrices=False)[0]

        for cell in range(num_cells_gen_ITI_F):
            
            # F-stats for NULL data
            # ---------------------
            # Get RSS with SVD approach
            y_neu = Y_null[:, cell]
            rss = np.sum(y_neu**2) - np.sum((np.transpose(U) @ y_neu)**2)
            rss_red = np.sum(y_neu**2) - np.sum((np.transpose(U_red) @ y_neu)**2)
            f_stat_rand = ((rss_red - rss) / m_k) / (rss / (T-M))

            # Store F_stat
            f_stats_ITIs_allcells[cell, k] = f_stat_rand
            
            # F-stats for OBS data
            # --------------------
            y_neu = Y_full[:, cell]
            rss = np.sum(y_neu**2) - np.sum((np.transpose(U) @ y_neu)**2)
            rss_red = np.sum(y_neu**2) - np.sum((np.transpose(U_red) @ y_neu)**2)
            f_stat_rand = ((rss_red - rss) / m_k) / (rss / (T-M))

            # Store F_stat
            f_stats_observed[cell, k] = f_stat_rand
            
            # EXTRA: also get delta R2
            tss = np.sum((y_neu - np.mean(y_neu))**2) # total sum of squares
            delta_r2[cell, k] = (1.0 - (rss / tss)) - (1.0 - (rss_red / tss))
    
    # Turn F-statistics into p-values and return them
    return_pvals = np.empty((num_cells, len(X_full)))
    for cell in range(num_cells):
        for k in range(K):
            return_pvals[cell, k] = (np.sum(f_stats_ITIs_allcells[:, k] >= f_stats_observed[cell, k]) + 1) / (f_stats_ITIs_allcells.shape[0] + 1)
    
    return((return_pvals, delta_r2))

# Compute the Results 
This function returns 2 elements:
- <b>pvals_wrap</b>= the matrix of the p-value for each individual element. Row= cell and column= predictor variable (in order)
- <b>delta_r2_wrap</b> = the change in the predictiveness of the full model and the model with the individual predictor removed. Row= cell and column= predictor variable (in order)

In [7]:
pvals_wrap, delta_r2_wrap = compute_pvalues_null_data(X_list=X_full, Y_full=Y_all, Y_null=Y_null_wrap)

In [8]:
#  Calculate coding proportions at p<0.01 threshold
n=Y_all.shape[1] # total number of cells
print('Proportion Coding')
print('Cue:' , np.array(np.where(pvals_wrap[:,0]<0.01)).shape[1]/n)
print('Lick:' , np.array(np.where(pvals_wrap[:,1]<0.01)).shape[1]/n)
print('R:' , np.array(np.where(pvals_wrap[:,2]<0.01)).shape[1]/n)
print('NR:' , np.array(np.where(pvals_wrap[:,3]<0.01)).shape[1]/n)
print('Value:' , np.array(np.where(pvals_wrap[:,4]<0.01)).shape[1]/n)
print('RPE:' , np.array(np.where(pvals_wrap[:,5]<0.01)).shape[1]/n)
print('CD:' , np.array(np.where(pvals_wrap[:,6]<0.01)).shape[1]/n)
print('CE:' , np.array(np.where(pvals_wrap[:,7]<0.01)).shape[1]/n)

Proportion Coding
Cue: 0.06825396825396825
Lick: 0.6111111111111112
R: 0.37936507936507935
NR: 0.3761904761904762
Value: 0.13015873015873017
RPE: 0.11904761904761904
CD: 0.1873015873015873
CE: 0.04126984126984127
